# Extração SISREG / e-SUS Regulação — Espera e acesso a especialidades para o projeto Conecta Saúde

Este notebook documenta a fonte **SISREG / e-SUS Regulação** para o projeto.

O objetivo é mostrar:

1. **quais dados seriam extraídos**;
2. **qual é a situação de acesso à API**;
3. **como preparar um conector quando houver acesso institucional**;
4. **como usar CSV anonimizado como alternativa para o MVP acadêmico**;
5. **quais perguntas de negócio podem ser respondidas com esses dados**.

> Ponto importante: diferente de SIH/SUS, SIA/SUS, CNES e IBGE, os microdados operacionais de regulação envolvem informações sensíveis sobre solicitações, agendamentos e acesso assistencial. Por isso, a API do SISREG/e-SUS Regulação não deve ser tratada como API pública aberta. Para o MVP acadêmico, a abordagem segura é usar dados agregados, anonimizados ou um CSV auxiliar sem CPF, CNS, nome, telefone ou endereço.


## 1. Papel do SISREG / e-SUS Regulação no projeto

Essa é a fonte mais aderente ao **Tema 4 — Gestão de Espera e Acesso a Especialidades**, porque permite analisar a jornada regulatória:

- solicitação de consulta, exame, procedimento ou internação;
- data da solicitação;
- prioridade/risco;
- especialidade ou procedimento solicitado;
- município de origem;
- unidade solicitante;
- unidade executante;
- status da solicitação;
- data de autorização;
- data de agendamento;
- data de execução;
- cancelamentos, faltas e pendências.

No projeto, essa fonte deve ser considerada como:

- **fonte ideal complementar**, caso exista acesso institucional;
- **evolução futura**, caso o grupo não tenha acesso real;
- **CSV anonimizado/sintético**, caso seja necessário demonstrar o conceito no MVP.


## 2. Situação da API e link de referência

Referências oficiais e institucionais:

```text
https://wiki.saude.gov.br/SISREG/index.php/P%C3%A1gina_principal
```

```text
https://wiki.saude.gov.br/e-SUSREGULACAO/index.php/P%C3%A1gina_principal
```

Portal de serviços do DATASUS:

```text
https://servicos-datasus.saude.gov.br/
```

Ponto de atenção para o projeto:

- o acesso à API operacional de regulação é **restrito** e depende de autorização institucional;
- os dados podem conter informações sensíveis de pacientes;
- para apresentação acadêmica, deve-se trabalhar com dados agregados, anonimizados ou sintéticos;
- o notebook abaixo entrega um padrão de conector para quando houver uma API autorizada e, em paralelo, uma ingestão via CSV auxiliar para viabilizar o MVP.


## 3. Dados que seriam extraídos

| Campo | Uso no projeto | Observação de privacidade |
|---|---|---|
| `id_solicitacao` | Identificar solicitação sem usar dados pessoais | Deve ser hash ou identificador técnico |
| `data_solicitacao` | Início da espera | Não identifica diretamente o paciente |
| `data_autorizacao` | Tempo até autorização | Usar apenas para cálculo agregado |
| `data_agendamento` | Tempo até marcação | Usar apenas para cálculo agregado |
| `data_execucao` | Tempo total até atendimento | Usar apenas para cálculo agregado |
| `status_solicitacao` | Pendente, autorizada, agendada, executada, cancelada | Sem dados pessoais |
| `prioridade` | Urgente, alta, média, baixa | Sem dados pessoais |
| `cod_municipio_origem` | Origem territorial da demanda | Código territorial |
| `cod_cnes_solicitante` | Unidade que solicitou | Código CNES |
| `cod_cnes_executante` | Unidade que executou | Código CNES |
| `cod_procedimento` | Procedimento ou especialidade solicitada | Enriquecer com SIGTAP |
| `especialidade` | Nome da especialidade | Sem dados pessoais |
| `tipo_regulacao` | Ambulatorial, exame, internação | Sem dados pessoais |

Campos que **não devem ser usados no MVP acadêmico**:

- CPF;
- CNS;
- nome do paciente;
- telefone;
- endereço;
- data de nascimento completa;
- observações clínicas livres;
- qualquer texto que possa identificar uma pessoa.


In [ ]:
# Instalação das dependências
# Execute esta célula no Colab/Jupyter caso os pacotes ainda não estejam instalados.

%pip install -q pandas pyarrow requests python-dotenv


## 4. Parâmetros de conexão autorizada

Abaixo está um modelo de conector REST genérico. Ele só deve ser usado se a equipe tiver um endpoint institucional autorizado.

As credenciais não devem ser escritas diretamente no notebook. Use variáveis de ambiente ou um arquivo `.env` fora do GitHub.


In [ ]:
from pathlib import Path
import os
import pandas as pd
import requests

DIRETORIO_RAW = Path("dados/raw/regulacao")
DIRETORIO_TRATADO = Path("dados/tratado/regulacao")

DIRETORIO_RAW.mkdir(parents=True, exist_ok=True)
DIRETORIO_TRATADO.mkdir(parents=True, exist_ok=True)

# Endpoint institucional autorizado. Exemplo fictício.
# Não existe, neste notebook, uma URL pública aberta de microdados para consulta livre.
API_REGULACAO_BASE_URL = os.getenv("API_REGULACAO_BASE_URL", "")
API_REGULACAO_TOKEN = os.getenv("API_REGULACAO_TOKEN", "")

print("API configurada?", bool(API_REGULACAO_BASE_URL and API_REGULACAO_TOKEN))


## 5. Função genérica de extração por API autorizada

Esta função é um modelo. O contrato real dependerá da API fornecida pela Secretaria de Saúde ou pelo ambiente institucional autorizado.


In [ ]:
def extrair_regulacao_api(
    data_inicio: str,
    data_fim: str,
    municipio: str | None = None,
    status: str | None = None,
    pagina: int = 1,
    tamanho_pagina: int = 1000,
) -> pd.DataFrame:
    '''
    Modelo de extração para API autorizada de regulação.

    Este código não deve ser executado sem um endpoint real autorizado.
    '''
    if not API_REGULACAO_BASE_URL or not API_REGULACAO_TOKEN:
        raise RuntimeError("Configure API_REGULACAO_BASE_URL e API_REGULACAO_TOKEN como variáveis de ambiente.")

    endpoint = f"{API_REGULACAO_BASE_URL.rstrip('/')}/solicitacoes"

    headers = {
        "Authorization": f"Bearer {API_REGULACAO_TOKEN}",
        "Accept": "application/json",
    }

    params = {
        "data_inicio": data_inicio,
        "data_fim": data_fim,
        "pagina": pagina,
        "tamanho_pagina": tamanho_pagina,
    }

    if municipio:
        params["municipio"] = municipio
    if status:
        params["status"] = status

    resposta = requests.get(endpoint, headers=headers, params=params, timeout=120)
    resposta.raise_for_status()

    payload = resposta.json()

    # Ajustar conforme contrato da API real.
    registros = payload.get("dados", payload if isinstance(payload, list) else [])
    return pd.DataFrame(registros)


## 6. Alternativa para o MVP: CSV anonimizado ou sintético

Para a Sprint, a opção mais segura é trabalhar com um CSV auxiliar no seguinte formato:

```text
id_solicitacao,data_solicitacao,data_autorizacao,data_agendamento,data_execucao,status_solicitacao,prioridade,cod_municipio_origem,cod_cnes_solicitante,cod_cnes_executante,cod_procedimento,especialidade,tipo_regulacao
```

A célula abaixo cria uma amostra sintética para validar as regras de negócio sem expor dados sensíveis.


In [ ]:
dados_exemplo = [
    {
        "id_solicitacao": "REQ0001",
        "data_solicitacao": "2024-01-03",
        "data_autorizacao": "2024-01-07",
        "data_agendamento": "2024-01-20",
        "data_execucao": "2024-02-02",
        "status_solicitacao": "executada",
        "prioridade": "alta",
        "cod_municipio_origem": "3550308",
        "cod_cnes_solicitante": "0000001",
        "cod_cnes_executante": "0000002",
        "cod_procedimento": "0301010072",
        "especialidade": "Cardiologia",
        "tipo_regulacao": "consulta",
    },
    {
        "id_solicitacao": "REQ0002",
        "data_solicitacao": "2024-01-05",
        "data_autorizacao": "2024-01-10",
        "data_agendamento": None,
        "data_execucao": None,
        "status_solicitacao": "pendente",
        "prioridade": "media",
        "cod_municipio_origem": "3550308",
        "cod_cnes_solicitante": "0000001",
        "cod_cnes_executante": None,
        "cod_procedimento": "0205010032",
        "especialidade": "Imagem",
        "tipo_regulacao": "exame",
    },
    {
        "id_solicitacao": "REQ0003",
        "data_solicitacao": "2024-01-08",
        "data_autorizacao": "2024-01-15",
        "data_agendamento": "2024-02-10",
        "data_execucao": None,
        "status_solicitacao": "agendada",
        "prioridade": "baixa",
        "cod_municipio_origem": "3509502",
        "cod_cnes_solicitante": "0000003",
        "cod_cnes_executante": "0000004",
        "cod_procedimento": "0301060061",
        "especialidade": "Ortopedia",
        "tipo_regulacao": "consulta",
    },
]

df_regulacao_raw = pd.DataFrame(dados_exemplo)

arquivo_csv_exemplo = DIRETORIO_RAW / "regulacao_exemplo_anonimizado.csv"
df_regulacao_raw.to_csv(arquivo_csv_exemplo, index=False)

print(f"CSV de exemplo salvo em: {arquivo_csv_exemplo}")
df_regulacao_raw


## 7. Leitura de CSV anonimizado

Quando o grupo tiver uma base auxiliar, substitua o caminho abaixo pelo arquivo real.


In [ ]:
CAMINHO_CSV_REGULACAO = arquivo_csv_exemplo  # Substituir pelo CSV real anonimizado.

df_regulacao = pd.read_csv(CAMINHO_CSV_REGULACAO, dtype=str)
df_regulacao.head()


## 8. Tratamento das datas e cálculo de tempos de espera


In [ ]:
def tratar_regulacao(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    campos_data = ["data_solicitacao", "data_autorizacao", "data_agendamento", "data_execucao"]
    for col in campos_data:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    if {"data_solicitacao", "data_autorizacao"}.issubset(df.columns):
        df["dias_ate_autorizacao"] = (df["data_autorizacao"] - df["data_solicitacao"]).dt.days

    if {"data_solicitacao", "data_agendamento"}.issubset(df.columns):
        df["dias_ate_agendamento"] = (df["data_agendamento"] - df["data_solicitacao"]).dt.days

    if {"data_solicitacao", "data_execucao"}.issubset(df.columns):
        df["dias_ate_execucao"] = (df["data_execucao"] - df["data_solicitacao"]).dt.days

    if "data_solicitacao" in df.columns:
        df["competencia_solicitacao"] = df["data_solicitacao"].dt.to_period("M").astype(str)

    return df


df_regulacao = tratar_regulacao(df_regulacao)
df_regulacao


## 9. Quais dados serão extraídos

| Campo tratado | Uso no projeto |
|---|---|
| `id_solicitacao` | Contagem de solicitações sem identificação pessoal |
| `competencia_solicitacao` | Evolução mensal da demanda regulada |
| `status_solicitacao` | Fila pendente, agendada, executada, cancelada |
| `prioridade` | Priorização por risco |
| `cod_municipio_origem` | Origem territorial da demanda |
| `cod_cnes_solicitante` | Unidade solicitante |
| `cod_cnes_executante` | Unidade executante |
| `cod_procedimento` | Enriquecimento com SIGTAP |
| `especialidade` | Ranking de especialidades sob pressão |
| `tipo_regulacao` | Consulta, exame, internação ou procedimento |
| `dias_ate_autorizacao` | Tempo até autorização |
| `dias_ate_agendamento` | Tempo até marcação |
| `dias_ate_execucao` | Tempo total até atendimento |


## 10. Salvamento da base tratada


In [ ]:
arquivo_saida = DIRETORIO_TRATADO / "regulacao_tratada_anonimizada.parquet"
df_regulacao.to_parquet(arquivo_saida, index=False)
print(f"Arquivo salvo em: {arquivo_saida}")


## 11. Perguntas que conseguimos responder com SISREG/e-SUS Regulação

Com a base de regulação, conseguimos responder perguntas diretamente ligadas ao Tema 4:

1. Qual é o tempo médio de espera por especialidade?
2. Quais especialidades têm maior fila pendente?
3. Quais municípios mais geram solicitações de regulação?
4. Quais unidades solicitantes concentram maior demanda?
5. Quais unidades executantes recebem maior volume de agendamentos?
6. Qual é o tempo médio entre solicitação e autorização?
7. Qual é o tempo médio entre solicitação e agendamento?
8. Qual é o tempo total até execução do atendimento?
9. Quais solicitações de alta prioridade estão demorando mais?
10. Quais tipos de regulação concentram maior gargalo: consulta, exame, procedimento ou internação?
11. Qual é a taxa de cancelamento ou não execução?
12. Quais especialidades precisam de ampliação de agenda ou redistribuição regional?


## 12. Exemplo 1 — Fila por especialidade e status


In [ ]:
fila_especialidade = (
    df_regulacao
    .groupby(["especialidade", "status_solicitacao"], dropna=False)
    .agg(qtd_solicitacoes=("id_solicitacao", "nunique"))
    .reset_index()
    .sort_values("qtd_solicitacoes", ascending=False)
)

fila_especialidade


## 13. Exemplo 2 — Tempo médio de espera por especialidade


In [ ]:
tempo_espera_especialidade = (
    df_regulacao
    .groupby("especialidade", dropna=False)
    .agg(
        qtd_solicitacoes=("id_solicitacao", "nunique"),
        media_dias_ate_autorizacao=("dias_ate_autorizacao", "mean"),
        media_dias_ate_agendamento=("dias_ate_agendamento", "mean"),
        media_dias_ate_execucao=("dias_ate_execucao", "mean"),
    )
    .reset_index()
    .sort_values("qtd_solicitacoes", ascending=False)
)

tempo_espera_especialidade


## 14. Exemplo 3 — Solicitações por município de origem


In [ ]:
solicitacoes_municipio = (
    df_regulacao
    .groupby("cod_municipio_origem", dropna=False)
    .agg(
        qtd_solicitacoes=("id_solicitacao", "nunique"),
        qtd_especialidades=("especialidade", "nunique"),
        media_dias_ate_agendamento=("dias_ate_agendamento", "mean"),
    )
    .reset_index()
    .sort_values("qtd_solicitacoes", ascending=False)
)

solicitacoes_municipio


## 15. Como essa fonte entra no painel

Com SISREG/e-SUS Regulação, o projeto deixa de trabalhar apenas com **produção realizada** e passa a enxergar a **demanda reprimida**.

Indicadores centrais:

```text
Tempo médio de espera = data_execucao - data_solicitacao
```

```text
Fila pendente por especialidade = solicitações com status pendente ou autorizada sem execução
```

```text
Taxa de execução = solicitações executadas / total de solicitações
```

```text
Taxa de cancelamento = solicitações canceladas / total de solicitações
```

```text
Pressão de acesso = fila pendente + tempo médio de espera + crescimento de solicitações
```

Esse conjunto de indicadores complementa o Tema 1, porque mostra não apenas onde a rede está internando mais, mas onde há gargalo antes do atendimento acontecer.
